# RSNA Knee — Baseline_v6 Inference

Based on v5, preserving its group → series → slot aggregation and label-query head.

- All series in each slot share a total window budget (default 32).
- Balanced quotas, capped at each series' original slice count; redistribute unused quota.
- Unique anchors at physical-depth targets within each series.
- Original [i-1, i, i+1] neighbors, repeating endpoints at boundaries.
- Keep v5 full-series intensity statistics, 336 px and 140 mm defaults.
- Pack only valid windows; no repeated anchors to fill a short slot.
- Budget, image size, crop and metadata switch are read from the v6 checkpoint.

Attach competition data, DINOv2 Small and **Baseline_v6** best.pt. Set CHECKPOINT_PATH in the final cell if multiple best.pt files are attached.


In [ ]:
"""RSNA Knee 数据加载与 140 mm physical crop 预处理模块。"""

from pathlib import Path

import numpy as np
import pandas as pd
import pydicom
from pydicom.errors import InvalidDicomError
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset


DEFAULT_ROOT = Path("/root/RSNA/rsna-knee-abnormality-detection")
DEFAULT_IMAGE_SIZE = 336
DEFAULT_CROP_MM = 140.0
DEFAULT_SLOT_WINDOW_BUDGET = 32
# 一个 Study 预测 12 个标签；SLOTS 固定六种扫描方向/脂肪抑制组合。
LABELS = ["ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA", "Lateral OA", "PF OA", "Effusion", "Synovitis", "Baker's", "Contusion", "Fracture"]
SLOTS = ["Axial_FS", "Axial_nonFS", "Coronal_FS", "Coronal_nonFS", "Sagittal_FS", "Sagittal_nonFS"]
# 每个序列对应 11 个数值元数据，顺序必须与 load_series 构造 features 的顺序一致。
SERIES_FEATURES = ["fluid_sensitive", "fat_suppression", "t1", "pd", "t2", "contrast_unknown", "te", "tr", "pixel_spacing", "slice_spacing", "coverage"] #翻译：
#MRI 序列特征：      液体敏感；         脂肪抑制； T1 加权；质子密度加权；T2 加权；对比或加权类型未知；回波时间；重复时间；像素间距；相邻切片间距；扫描覆盖范围。

def _prepare_series_df(series_df, image_root):
    # 保留全部序列，不再为择一规则扫描切片数。
    series_df = series_df.copy()
    series_df["SeriesPath"] = [Path(image_root) / str(study) / str(series) for study, series in zip(series_df.StudyInstanceUID, series_df.SeriesInstanceUID)]
    series_df["SeriesSlot"] = series_df.Anatomical_Plane + "_" + np.where(series_df.Fat_Suppression.eq(1), "FS", "nonFS")
    return series_df


def load_metadata(root=DEFAULT_ROOT):
    # Study 标签表与序列描述表分别读取；实际 DICOM 只在 __getitem__ 时按需解码。
    root = Path(root)
    train_df, test_df = pd.read_csv(root / "train.csv"), pd.read_csv(root / "test.csv")
    train_series_df = _prepare_series_df(pd.read_csv(root / "train_series.csv"), root / "train_series")
    test_series_df = _prepare_series_df(pd.read_csv(root / "test_series.csv"), root / "test_series")
    return train_df, train_series_df, test_df, test_series_df, pd.read_csv(root / "sample_submission.csv")


def get_sorted_dicom_info(series_path):

    """使用患者坐标排序，并把主切片方向统一为患者坐标轴的正方向。"""

    # 获取当前MRI序列文件夹下所有DICOM文件路径
    files = list(Path(series_path).glob("*.dcm"))
    if not files:
        raise FileNotFoundError(f"No DICOM files found in series: {series_path}")
    # 读取所有DICOM文件的头信息，但不读取像素数据，节省内存
    headers = [pydicom.dcmread(file, stop_before_pixels=True) for file in files]
    # 取第一张切片的ImageOrientationPatient，表示该图片在患者空间中的行方向和列方向   #ImagePositionPatient表示这一张图片左上角在患者三维空间的位置
    orientation = np.asarray(headers[0].ImageOrientationPatient, dtype=np.float32)
    # 前3个数表示图片行方向，后3个数表示图片列方向，两个方向叉乘得到切片法向量
    normal = np.cross(orientation[:3], orientation[3:])
    # 将法向量归一化，使其长度变为1，方便后续进行空间投影计算
    normal /= np.linalg.norm(normal)
    # 判断法向量主要方向是否为负，如果是则翻转，保证所有序列方向统一
    if normal[np.argmax(np.abs(normal))] < 0:
        # 反转法向量方向
        normal = -normal
    # 将每张切片的患者空间坐标投影到切片法向量方向，得到每张slice的真实空间位置
    positions = np.asarray([np.dot(np.asarray(header.ImagePositionPatient, dtype=np.float32), normal) for header in headers])
    # 根据真实空间位置从小到大排序，得到正确的slice顺序
    order = np.argsort(positions)
    # 按照排序后的索引，同时返回：
    # 1. 排序后的DICOM文件 2. 对应空间位置 3. 对应header信息
    return [files[i] for i in order], positions[order], [headers[i] for i in order]


def sort_dicom_files(series_path):
    # 只需要已排序文件列表时的便捷包装。
    return get_sorted_dicom_info(series_path)[0]

#读取一张DICOM MRI切片，将其转换成标准的二维numpy图像矩阵，并处理少数异常DICOM文件和灰度方向问题。
def read_dicom(path):
    dicom = pydicom.dcmread(path)
    # 尝试直接通过pydicom解析PixelData得到图像矩阵
    try:
        # pixel_array会自动根据DICOM头信息解析像素，并转换成numpy数组
        image = dicom.pixel_array.astype(np.float32)
    # 如果标准解析失败，进入手动恢复流程
    except ValueError as error:
        # 获取图像高度和宽度，后续用于手动reshape像素数据
        rows, columns = int(dicom.Rows), int(dicom.Columns)
        # 获取帧数，多帧DICOM没有该字段时默认认为只有1帧
        frames = int(getattr(dicom, "NumberOfFrames", 1) or 1)
        # 获取每个像素包含几个采样值，例如RGB可能是3，MRI通常是1
        samples = int(getattr(dicom, "SamplesPerPixel", 1) or 1)
        # 计算理论上的像素总数量
        pixel_count = rows * columns * frames * samples
        # 直接读取DICOM中的原始像素字节数据
        raw = dicom.PixelData
        # 获取DICOM传输格式，用于判断PixelData是否经过压缩
        transfer_syntax = getattr(getattr(dicom, "file_meta", None), "TransferSyntaxUID", None)
        # 判断当前DICOM是否为未压缩格式
        uncompressed = transfer_syntax is None or not transfer_syntax.is_compressed

        # 只有满足以下条件才允许手动解析：
        # 1. 未压缩
        # 2. 单帧
        # 3. 单通道
        # 4. 原始字节数量符合预期
        if not uncompressed or frames != 1 or samples != 1 or len(raw) not in (pixel_count, pixel_count + 1):

            # 如果情况复杂，无法安全恢复，则直接抛出异常
            raise ValueError(f"Cannot decode DICOM {path}: {error}") from error

        # 根据PixelRepresentation判断像素是否为有符号整数
        # 0代表unsigned，1代表signed
        dtype = np.int8 if int(getattr(dicom, "PixelRepresentation", 0) or 0) else np.uint8

        # 从原始字节创建numpy数组，并恢复成二维图像
        image = np.frombuffer(raw[:pixel_count], dtype=dtype).reshape(rows, columns).astype(np.float32)

    # 根据DICOM标准应用灰度线性变换
    # 输出值 = 原始像素值 × RescaleSlope + RescaleIntercept
    image = image * float(getattr(dicom, "RescaleSlope", 1)) + float(getattr(dicom, "RescaleIntercept", 0))

    # MONOCHROME1表示像素越大越黑，需要反转成普通医学图像亮度方向
    if getattr(dicom, "PhotometricInterpretation", "") == "MONOCHROME1":
        # 使用最大值+最小值减去原像素，实现灰度反转
        image = image.max() + image.min() - image

    # 返回最终处理好的二维MRI切片
    return image

#读取指定的MRI切片，并且在遇到损坏DICOM时，用距离最近的正常切片替代，最后把所有slice组合成3D volume。注意这里的volume是一个numpy数组，shape为[S,H,W]，其中S是切片数，H和W是每张切片的高度和宽度。不是channle为3
def _read_sampled_slices(files, indices):
    """读取采样切片；单张损坏时用空间上最近的可读切片替代。"""
    # cache 避免重复解码同一张；invalid 避免对已知坏文件反复尝试。
    cache, invalid = {}, set()
    images = []
    for index in indices:
        index = int(index)
        # 按切片索引距离从近到远尝试，直到找到可解码的替代切片。
        candidates = sorted(range(len(files)), key=lambda candidate: abs(candidate - index))
        for candidate in candidates:
            if candidate in cache:
                image = cache[candidate]
                break
            if candidate in invalid:
                continue
            try:
                image = read_dicom(files[candidate])
                cache[candidate] = image
                break
            except (ValueError, OSError, EOFError, InvalidDicomError):
                invalid.add(candidate)
        else:
            raise RuntimeError(f"No decodable DICOM slices in series: {Path(files[0]).parent}")
        images.append(image)
    return np.stack(images)


def allocate_slot_windows(slice_counts, budget=DEFAULT_SLOT_WINDOW_BUDGET):
    """均衡分配 slot 总窗口预算；每序列至少一组，且不超过原始切片数。"""
    capacities = np.asarray(slice_counts, dtype=np.int64)
    if budget < 1 or capacities.ndim != 1 or np.any(capacities < 1):
        raise ValueError("Window budget and series slice counts must be positive")
    if len(capacities) > budget:
        raise ValueError(f"Slot has {len(capacities)} series but window budget is {budget}; "
                         "increase --slot-window-budget to retain every series")
    quotas = np.zeros_like(capacities)
    remaining = min(int(capacities.sum()), budget)
    # 按 SeriesInstanceUID 排序后的序列轮流领取一个窗口；短序列用完后自动跳过。
    while remaining:
        for index, capacity in enumerate(capacities):
            if quotas[index] < capacity:
                quotas[index] += 1
                remaining -= 1
                if not remaining:
                    break
    return quotas


def select_unique_anchors(positions, count):
    """物理等间隔目标→不同原始索引；为后续 anchor 留足切片，避免最近邻重复。"""
    positions = np.asarray(positions)
    if positions.ndim != 1 or not 1 <= count <= len(positions):
        raise ValueError("Anchor count must be between one and the number of original slices")
    if not np.isfinite(positions).all() or np.any(np.diff(positions) < 0):
        raise ValueError("Slice positions must be finite and sorted")
    if positions[-1] == positions[0]:
        # 物理位置无法区分时按原始排序索引均匀选，不重复索引。
        return np.rint(np.linspace(0, len(positions) - 1, count)).astype(np.int64) if count > 1 else np.array([len(positions) // 2])
    targets = (np.linspace(positions[0], positions[-1], count) if count > 1
               else np.array([(positions[0] + positions[-1]) / 2]))
    anchors, lower = [], 0
    for index, target in enumerate(targets):
        upper = len(positions) - (count - index)
        chosen = lower + int(np.abs(positions[lower:upper + 1] - target).argmin())
        anchors.append(chosen)
        lower = chosen + 1
    return np.asarray(anchors, dtype=np.int64)


def make_adjacent_windows(volume, positions, num_windows):
    """每个不同 anchor 使用原始 [i-1,i,i+1]；仅序列边界重复端点。"""
    if not len(volume) or len(positions) != len(volume) or num_windows < 1:
        raise ValueError("Expected non-empty slices/positions and a positive window count")
    anchors = select_unique_anchors(positions, min(num_windows, len(volume)))
    indices = np.clip(anchors[:, None] + np.array([-1, 0, 1]), 0, len(volume) - 1)
    indices = torch.as_tensor(indices, dtype=torch.long, device=volume.device)
    images = volume[indices, 0]  # [窗口数,3,H,W]，不同序列绝不拼接成邻居。
    coverage = float(positions[-1] - positions[0])
    normalized = 2 * (np.asarray(positions)[anchors] - positions[0]) / coverage - 1 if coverage > 0 else np.zeros(len(anchors))
    return images, torch.as_tensor(normalized, dtype=torch.float32)


#从一个MRI序列的DICOM header中读取像素空间大小（PixelSpacing），用于后续把“毫米尺度的crop”转换成“像素尺度的crop”。
def _get_pixel_spacing(headers, series_path):

    """读取有效的行、列 PixelSpacing；优先使用堆栈中央切片。"""

    # 计算DICOM序列中间位置的slice索引，优先检查中间切片
    middle = len(headers) // 2
    # 按距离中间切片的远近排序，生成检查顺序
    search_order = sorted(range(len(headers)), key=lambda index: abs(index - middle))
    # 按照从中间到两边的顺序遍历每一张slice的header
    for index in search_order:

        # 获取当前slice的PixelSpacing字段，如果不存在则返回None
        value = getattr(headers[index], "PixelSpacing", None)

        # 如果当前slice没有PixelSpacing信息，跳过这一张slice
        if value is None or len(value) < 2:
            continue

        # 将PixelSpacing转换成numpy数组，只保留行方向和列方向的两个spacing值
        spacing = np.asarray(value[:2], dtype=np.float32)

        # 检查spacing是否有效：1. 所有值不是NaN/inf 2. 所有值大于0
        if np.all(np.isfinite(spacing)) and np.all(spacing > 0):
            # 返回有效的像素间距，单位通常是mm/pixel
            return spacing
    # 如果遍历所有slice都没有找到有效PixelSpacing，则说明该序列无法确定物理尺寸
    raise ValueError(f"Series has no valid PixelSpacing: {series_path}")


def physical_center_crop(volume, pixel_spacing, crop_mm=DEFAULT_CROP_MM):

    """按真实毫米尺度中心裁剪；原始 FOV 不足时先对称补零以保持尺度。"""

    # 检查输入volume是否为三维MRI数据，格式应该是 [slice数量, height, width]
    if volume.ndim != 3:
        # 如果不是三维数据，说明输入格式错误，直接报错
        raise ValueError(f"Expected volume [S,H,W], got shape {volume.shape}")
    # 检查裁剪尺寸是否有效，crop_mm必须是正数
    if not np.isfinite(crop_mm) or crop_mm <= 0:
        # 如果crop_mm为负数、0或者无穷/NaN，则无法进行裁剪
        raise ValueError(f"crop_mm must be positive, got {crop_mm}")
    # 将pixel_spacing转换成numpy数组，方便后续数学计算
    spacing = np.asarray(pixel_spacing, dtype=np.float32)

    # 检查PixelSpacing是否合法：必须包含两个值，并且不能为NaN、inf，同时两个方向都必须大于0
    if spacing.shape != (2,) or not np.all(np.isfinite(spacing)) or np.any(spacing <= 0):
        # PixelSpacing异常时无法从毫米转换到像素，因此停止处理
        raise ValueError(f"Invalid PixelSpacing: {pixel_spacing}")
    # 根据真实毫米大小计算需要裁剪的像素高度 # 公式：目标像素数量 = crop毫米数 / 每个像素代表的毫米数
    target_height = max(1, int(round(crop_mm / float(spacing[0]))))
    # 根据真实毫米大小计算需要裁剪的像素宽度
    target_width = max(1, int(round(crop_mm / float(spacing[1]))))
    # 获取当前MRI volume的空间尺寸
    # volume形状：[slice,height,width]
    height, width = volume.shape[-2:]

    # 计算高度方向需要补多少像素 # 如果原图高度已经大于目标高度，则不需要补
    pad_height = max(0, target_height - height)
    # 计算宽度方向需要补多少像素 # 如果原图宽度已经大于目标宽度，则不需要补
    pad_width = max(0, target_width - width)

    # 如果原始视野(FOV)小于目标裁剪区域，则先进行padding
    if pad_height or pad_width:

        # 原图不足140mm时进行左右/上下对称补零
        # 避免直接放大较小视野导致空间尺度错误
        volume = np.pad(
            volume,

            # 三个维度分别对应：
            # slice维度不补，height方向补，width方向补
            (
                (0, 0),
                # height方向上下对称补零
                (pad_height // 2, pad_height - pad_height // 2),
                # width方向左右对称补零
                (pad_width // 2, pad_width - pad_width // 2),
            ),

            # 使用0填充，代表MRI背景区域
            mode="constant",
            # 指定填充值为0
            constant_values=0,
        )
        # padding之后重新获取新的height和width
        height, width = volume.shape[-2:]

    # 计算中心裁剪开始位置的高度坐标
    top = (height - target_height) // 2

    # 计算中心裁剪开始位置的宽度坐标
    left = (width - target_width) // 2

    # 对所有slice使用同一个中心区域进行裁剪
    # 保证整个3D volume的空间位置保持一致
    return volume[:, top:top + target_height, left:left + target_width]


def _contrast_features(te, tr, fat_suppression):
    """依据 EDA 使用的 TE/TR 启发式生成 T1、PD、T2、未知四维标记。"""
    # 这些是经验规则推断的扫描对比度类别，不是人工标注的真值。
    if fat_suppression:
        return [0, 1, 1, 0]
    if te <= 0 or tr <= 0:
        return [0, 0, 0, 1]
    if tr < 1000 and te < 30:
        return [1, 0, 0, 0]
    if te < 60:
        return [0, 1, 0, 0]
    return [0, 0, 1, 0]


def load_series(
    series_path,
    image_size=DEFAULT_IMAGE_SIZE,
    crop_mm=DEFAULT_CROP_MM,
    fluid_sensitive=0,
    fat_suppression=0,
    return_features=False,
    num_windows=DEFAULT_SLOT_WINDOW_BUDGET,
    dicom_info=None,
):

    """沿用 v5 整序列强度统计；按配额选不同 anchor，取原始相邻三张。"""

    # 获取当前MRI序列所有DICOM文件，并按照患者真实空间位置排序
    files, positions, headers = get_sorted_dicom_info(series_path) if dicom_info is None else dicom_info
    # 每张原始切片读取一次；坏文件仍沿用 v1 的最近可读切片替代策略。
    volume = _read_sampled_slices(files, np.arange(len(files)))

    # 获取该MRI序列的像素间距(mm/pixel)，用于物理尺度裁剪
    spacing = _get_pixel_spacing(headers, series_path)
    # 按真实毫米尺寸进行中心裁剪，例如裁剪140mm×140mm区域
    volume = physical_center_crop(volume, spacing, crop_mm)

    # 提取volume中有效的非零像素，用于计算MRI强度范围   # 排除背景0值，避免背景影响归一化
    foreground = volume[np.isfinite(volume) & (volume != 0)]
    # 如果裁剪后没有有效像素，说明该序列异常，无法继续处理
    if not foreground.size:
        raise ValueError(f"Series contains no finite non-zero pixels after crop: {series_path}")
    # 计算前景像素的0.5%和99.5%分位数作为强度截断范围 # 避免极端噪声影响归一化
    low, high = np.percentile(foreground, [0.5, 99.5])
    # 将MRI强度归一化到0~1范围，并限制异常值
    volume = np.clip((volume - low) / (high - low + 1e-6), 0, 1).astype(np.float32)
    # 将numpy数组转换为PyTorch tensor
    volume = torch.from_numpy(volume)[:, None]
    # 只 resize 选中的窗口；整序列分位数、crop 和归一化与 v5 一致。
    images, group_positions = make_adjacent_windows(volume, positions, num_windows)
    images = F.interpolate(
        images,
        size=(image_size, image_size),
        mode="bilinear",
        align_corners=False
    )
    # 选择中间slice的header作为整个series的代表metadata
    header = headers[len(headers) // 2]
    # 获取Echo Time和Repetition Time
    # MRI序列的重要扫描参数
    te, tr = float(getattr(header, "EchoTime", 0) or 0), float(getattr(header, "RepetitionTime", 0) or 0)
    # 根据相邻slice空间位置差计算slice间距
    # 使用median减少异常slice影响
    slice_spacing = float(np.median(np.abs(np.diff(positions)))) if len(positions) > 1 else 0


    # 计算整个MRI序列覆盖的物理长度(mm) # 即第一张slice到最后一张slice之间的距离
    coverage = float(abs(positions[-1] - positions[0])) if len(positions) > 1 else 0
    # 根据TE/TR以及脂肪抑制信息，推断MRI对比类型
    # 输出T1、PD、T2、unknown等标记
    contrast = _contrast_features(te, tr, int(fat_suppression))
    # 将MRI序列相关信息组成11维feature向量
    # 顺序必须与SERIES_FEATURES保持一致
    features = torch.tensor(
        [
            float(fluid_sensitive),
            float(fat_suppression),
            *contrast,
            te / 100,
            tr / 5000,
            float(spacing.mean()),
            slice_spacing / 10,
            coverage / 200
        ],
        dtype=torch.float32
    )
    # 根据需求决定是否同时返回图像和metadata feature
    # 训练模型时可能需要两者
    return (images, features, group_positions) if return_features else images


class KneeDataset(Dataset):
    """每个 Study 保留六个 slot 的全部序列，图像按有效三张组合紧凑存放。"""

    def __init__(self, study_df, series_df, image_size=DEFAULT_IMAGE_SIZE, crop_mm=DEFAULT_CROP_MM,
                 slot_window_budget=DEFAULT_SLOT_WINDOW_BUDGET):
        if slot_window_budget < 1:
            raise ValueError("slot_window_budget must be positive")
        self.study_df = study_df.reset_index(drop=True)
        self.image_size, self.crop_mm = image_size, crop_mm
        self.slot_window_budget = slot_window_budget
        self.series_groups = {uid: group.reset_index(drop=True)
                              for uid, group in series_df.groupby("StudyInstanceUID")}

    def __len__(self):
        return len(self.study_df)

    def __getitem__(self, index):
        row = self.study_df.iloc[index]
        uid = row.StudyInstanceUID
        if uid not in self.series_groups:
            raise ValueError(f"No series metadata for study {uid}")
        series_df = self.series_groups[uid]
        images, metadata, positions, counts, slots = [], [], [], [], []
        slot_mask = torch.zeros(len(SLOTS), dtype=torch.bool)
        for slot_index, slot in enumerate(SLOTS):
            # 序列顺序只用于可复现；全部保留，不按 Fluid_Sensitive 或 NumSlices 择一。
            candidates = series_df[series_df.SeriesSlot == slot].sort_values("SeriesInstanceUID")
            selected_series = list(candidates.itertuples(index=False))
            # 先读取各序列排序后的头信息，分配配额；传给 loader 避免重复读取头信息。
            dicom_infos = [get_sorted_dicom_info(selected.SeriesPath) for selected in selected_series]
            quotas = allocate_slot_windows([len(info[0]) for info in dicom_infos], self.slot_window_budget)
            for selected, info, quota in zip(selected_series, dicom_infos, quotas):
                groups, features, group_positions = load_series(
                    selected.SeriesPath, image_size=self.image_size, crop_mm=self.crop_mm,
                    fluid_sensitive=selected.Fluid_Sensitive,
                    fat_suppression=selected.Fat_Suppression, return_features=True,
                    num_windows=int(quota), dicom_info=info,
                )
                images.append(groups)
                metadata.append(features)
                positions.append(group_positions)
                counts.append(len(groups))
                slots.append(slot_index)
                slot_mask[slot_index] = True
        if not images:
            raise ValueError(f"No supported MRI series for study {uid}")
        values = (pd.to_numeric(row[LABELS], errors="coerce").to_numpy(dtype=np.float32)
                  if set(LABELS).issubset(row.index) else np.full(len(LABELS), np.nan, dtype=np.float32))
        valid = ~np.isnan(values)
        return {
            "study_uid": uid,
            "images": torch.cat(images),                 # [所有序列组数之和,3,H,W]
            "group_positions": torch.cat(positions),    # 每个组在所属序列内的物理深度
            "group_counts": torch.tensor(counts, dtype=torch.long),
            "series_slot_indices": torch.tensor(slots, dtype=torch.long),
            "series_features": torch.stack(metadata),  # [序列数,11]
            "slot_mask": slot_mask,
            "targets": torch.from_numpy(np.nan_to_num(values, nan=0)),
            "label_mask": torch.from_numpy(valid.astype(np.float32)),
            "label_weight": torch.from_numpy(np.where(valid, 2 * np.abs(values - 0.5), 0).astype(np.float32)),
        }


def collate_studies(samples):
    """拼接有效图像/序列，保持每个序列的组边界以及所属 Study、slot。"""
    if not samples:
        raise ValueError("Cannot collate an empty batch")
    batch = {"study_uid": [sample["study_uid"] for sample in samples]}
    for key in ("images", "group_positions", "group_counts", "series_slot_indices", "series_features"):
        batch[key] = torch.cat([sample[key] for sample in samples])
    batch["series_batch_indices"] = torch.cat([
        torch.full((len(sample["group_counts"]),), index, dtype=torch.long)
        for index, sample in enumerate(samples)
    ])
    for key in ("slot_mask", "targets", "label_mask", "label_weight"):
        batch[key] = torch.stack([sample[key] for sample in samples])
    return batch


def build_datasets(root=DEFAULT_ROOT, image_size=DEFAULT_IMAGE_SIZE, crop_mm=DEFAULT_CROP_MM,
                   slot_window_budget=DEFAULT_SLOT_WINDOW_BUDGET):
    metadata = load_metadata(root)
    train_df, train_series_df, test_df, test_series_df, sample_submission = metadata
    train_dataset = KneeDataset(train_df, train_series_df, image_size, crop_mm, slot_window_budget)
    test_dataset = KneeDataset(test_df, test_series_df, image_size, crop_mm, slot_window_budget)
    return train_dataset, test_dataset, sample_submission, metadata


def build_dataloaders(root=DEFAULT_ROOT, batch_size=2, num_workers=4,
                      image_size=DEFAULT_IMAGE_SIZE, crop_mm=DEFAULT_CROP_MM,
                      slot_window_budget=DEFAULT_SLOT_WINDOW_BUDGET):
    train_dataset, test_dataset, sample_submission, metadata = build_datasets(root, image_size, crop_mm, slot_window_budget)
    kwargs = dict(batch_size=batch_size, num_workers=num_workers, pin_memory=True, collate_fn=collate_studies)
    return (DataLoader(train_dataset, shuffle=True, **kwargs),
            DataLoader(test_dataset, shuffle=False, **kwargs), sample_submission, metadata)


__all__ = [
    "DEFAULT_ROOT", "DEFAULT_IMAGE_SIZE", "DEFAULT_CROP_MM", "DEFAULT_SLOT_WINDOW_BUDGET", "LABELS", "SLOTS",
    "SERIES_FEATURES", "load_metadata", "get_sorted_dicom_info", "sort_dicom_files",
    "read_dicom", "physical_center_crop", "allocate_slot_windows", "select_unique_anchors", "make_adjacent_windows", "load_series",
    "KneeDataset", "collate_studies", "build_datasets", "build_dataloaders",
]


In [ ]:
"""沿用 v5 网络：slot 共享窗口预算、原始相邻三张输入、层次聚合。"""

from contextlib import nullcontext
from pathlib import Path

import torch
import torch.nn as nn



ARCHITECTURE = "baseline_v6_slot_budget_adjacent"


def load_backbone(model_dir):
    from transformers import AutoModel

    model_dir = Path(model_dir)
    required = [model_dir / "config.json", model_dir / "pytorch_model.bin"]
    missing = [path.name for path in required if not path.is_file()]
    if missing:
        raise FileNotFoundError(f"DINOv2 model directory {model_dir} is missing {missing}")
    return AutoModel.from_pretrained(str(model_dir), local_files_only=True)


class RSNADINOv2(nn.Module):
    def __init__(self, model_dir=None, hidden_dim=256, num_heads=8,
                 slice_layers=2, slot_layers=1, dropout=0.1,
                 freeze_backbone=True, encoder_chunk_size=24):
        super().__init__()
        if encoder_chunk_size < 1:
            raise ValueError("encoder_chunk_size must be positive")
        self.backbone = load_backbone(model_dir)
        self.encoder_chunk_size = encoder_chunk_size
        self.freeze_backbone(freeze_backbone)
        backbone_dim = self.backbone.config.hidden_size
        self.register_buffer("image_mean", torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1), persistent=False)
        self.register_buffer("image_std", torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1), persistent=False)
        self.slice_projection = nn.Sequential(nn.LayerNorm(backbone_dim), nn.Linear(backbone_dim, hidden_dim))
        # 输入所属序列内归一化的物理组中心，不依赖固定组数，也不把不同序列连成一条轴。
        self.group_position = nn.Sequential(nn.Linear(1, hidden_dim), nn.GELU(), nn.Linear(hidden_dim, hidden_dim))
        slice_layer = nn.TransformerEncoderLayer(hidden_dim, num_heads, hidden_dim * 4,
                                                dropout, batch_first=True, norm_first=True)
        self.slice_encoder = nn.TransformerEncoder(slice_layer, slice_layers, enable_nested_tensor=False)
        self.slice_attention = nn.Sequential(nn.LayerNorm(hidden_dim), nn.Linear(hidden_dim, 1))
        self.metadata_projection = nn.Sequential(
            nn.Linear(len(SERIES_FEATURES), hidden_dim), nn.LayerNorm(hidden_dim),
            nn.GELU(), nn.Dropout(dropout), nn.Linear(hidden_dim, hidden_dim),
        )
        self.slot_embedding = nn.Parameter(torch.zeros(1, len(SLOTS), hidden_dim))
        slot_layer = nn.TransformerEncoderLayer(hidden_dim, num_heads, hidden_dim * 4,
                                               dropout, batch_first=True, norm_first=True)
        self.slot_encoder = nn.TransformerEncoder(slot_layer, slot_layers, enable_nested_tensor=False)
        # 保留 v1 的逐标签 query、交叉注意力和输出头。
        self.label_queries = nn.Parameter(torch.empty(len(LABELS), hidden_dim))
        self.label_attention = nn.MultiheadAttention(hidden_dim, num_heads, dropout=dropout, batch_first=True)
        self.output_norm = nn.LayerNorm(hidden_dim)
        self.label_weight = nn.Parameter(torch.empty(len(LABELS), hidden_dim))
        self.label_bias = nn.Parameter(torch.zeros(len(LABELS)))
        for parameter in (self.slot_embedding, self.label_queries, self.label_weight):
            nn.init.trunc_normal_(parameter, std=0.02)

    def freeze_backbone(self, freeze=True):
        for parameter in self.backbone.parameters():
            parameter.requires_grad = not freeze

    def unfreeze_last_blocks(self, num_blocks=2):
        self.freeze_backbone(True)
        for block in self.backbone.encoder.layer[-num_blocks:]:
            for parameter in block.parameters():
                parameter.requires_grad = True
        for parameter in self.backbone.layernorm.parameters():
            parameter.requires_grad = True

    def encode_images(self, images):
        """输入仅包含有效组 [总组数,3,H,W]；图像不存在 padding。"""
        use_grad = any(parameter.requires_grad for parameter in self.backbone.parameters())
        with nullcontext() if use_grad else torch.no_grad():
            # 每个 chunk 单独归一化，避免额外创建一整份归一化图像。
            chunks = [self.backbone(pixel_values=(chunk - self.image_mean) / self.image_std).last_hidden_state[:, 0]
                      for chunk in images.split(self.encoder_chunk_size)]
        return torch.cat(chunks)

    def aggregate_groups(self, features, group_counts, group_positions):
        """只对特征补齐；每行是一个真实序列，保证至少有一个有效组。"""
        if group_counts.ndim != 1 or not len(group_counts) or bool((group_counts <= 0).any()):
            raise ValueError("Every series must contain at least one group")
        if int(group_counts.sum()) != len(features) or len(group_positions) != len(features):
            raise ValueError("Group counts/positions do not match packed image features")
        series_count, max_groups = len(group_counts), int(group_counts.max())
        starts = group_counts.cumsum(0) - group_counts
        series_indices = torch.repeat_interleave(torch.arange(series_count, device=features.device), group_counts)
        local_indices = torch.arange(len(features), device=features.device) - torch.repeat_interleave(starts, group_counts)
        flat_indices = series_indices * max_groups + local_indices
        positioned = features + self.group_position(group_positions.to(features.dtype).unsqueeze(-1))
        padded = features.new_zeros(series_count * max_groups, features.shape[-1])
        padded = padded.index_copy(0, flat_indices, positioned).view(series_count, max_groups, -1)
        group_mask = torch.arange(max_groups, device=features.device)[None, :] < group_counts[:, None]
        padded = self.slice_encoder(padded, src_key_padding_mask=~group_mask)
        scores = self.slice_attention(padded).squeeze(-1).masked_fill(~group_mask, -torch.inf)
        attention = scores.softmax(dim=1)
        return (padded * attention.unsqueeze(-1)).sum(dim=1)

    @staticmethod
    def aggregate_series(series_features, series_batch_indices, series_slot_indices, batch_size):
        """同 slot 内按真实序列平均；每个序列先聚合为一个向量，长序列不会因组数多占权重。"""
        indices = series_batch_indices * len(SLOTS) + series_slot_indices
        sums = series_features.new_zeros(batch_size * len(SLOTS), series_features.shape[-1])
        sums = sums.index_add(0, indices, series_features)
        counts = series_features.new_zeros(batch_size * len(SLOTS))
        counts = counts.index_add(0, indices, series_features.new_ones(len(series_features)))
        return (sums / counts.clamp_min(1).unsqueeze(-1)).view(batch_size, len(SLOTS), -1)

    def forward(self, images, slot_mask, series_features=None, *, group_counts,
                group_positions, series_batch_indices, series_slot_indices):
        if slot_mask.ndim != 2 or slot_mask.shape[1] != len(SLOTS) or not bool(slot_mask.bool().any(dim=1).all()):
            raise ValueError("Every study must contain at least one valid slot")
        batch_size = len(slot_mask)
        features = self.slice_projection(self.encode_images(images))
        encoded_series = self.aggregate_groups(features, group_counts, group_positions)
        if series_features is not None:
            encoded_series = encoded_series + self.metadata_projection(series_features)
        slot_features = self.aggregate_series(encoded_series, series_batch_indices, series_slot_indices, batch_size)
        slot_features = self.slot_encoder(slot_features + self.slot_embedding,
                                         src_key_padding_mask=~slot_mask.bool())
        queries = self.label_queries.unsqueeze(0).expand(batch_size, -1, -1)
        label_features, _ = self.label_attention(queries, slot_features, slot_features,
                                                key_padding_mask=~slot_mask.bool(), need_weights=False)
        label_features = self.output_norm(label_features)
        return (label_features * self.label_weight.unsqueeze(0)).sum(dim=-1) + self.label_bias


def predict_batch(model, batch, use_metadata=True):
    """训练/验证/推理统一调用接口；batch 的张量应事先移动到目标设备。"""
    return model(batch["images"], batch["slot_mask"],
                 batch["series_features"] if use_metadata else None,
                 group_counts=batch["group_counts"], group_positions=batch["group_positions"],
                 series_batch_indices=batch["series_batch_indices"],
                 series_slot_indices=batch["series_slot_indices"])


__all__ = ["ARCHITECTURE", "RSNADINOv2", "predict_batch"]


In [ ]:
"""Kaggle submission inference for checkpoints trained by this package."""

import os
import time
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

os.environ.setdefault("HF_HUB_DISABLE_PROGRESS_BARS", "1")
os.environ.setdefault("TRANSFORMERS_NO_ADVISORY_WARNINGS", "1")

import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader




# Kaggle input paths confirmed by the successful run; edit here if mounts change.
DATA_ROOT = "/kaggle/input/competitions/rsna-knee-abnormality-detection"
CHECKPOINT_PATH = ""  # Auto-discover a unique best.pt; attach Baseline_v6 weights or set this path.
DINOV2_MODEL_DIR = "/kaggle/input/models/metaresearch/dinov2/pytorch/small/1"
OUTPUT_PATH = Path("/kaggle/working/submission.csv")
BATCH_SIZE = 1
NUM_WORKERS = 0
ENCODER_CHUNK_SIZE = 16
# Use up to two Kaggle GPUs; a single GPU works too.
NUM_GPUS = min(2, torch.cuda.device_count())


def find_one(candidates, description):
    paths = sorted(set(Path(path) for path in candidates))
    if len(paths) != 1:
        raise RuntimeError(f"Expected one {description}, found {len(paths)}: {paths}. Set its path explicitly above.")
    return paths[0]


def input_files():
    """Find small input metadata without descending into millions of DICOM files."""
    for directory, subdirs, files in os.walk("/kaggle/input"):
        subdirs[:] = [name for name in subdirs if name not in {"train_series", "test_series"}]
        here = Path(directory)
        for name in files:
            if name in {"test_series.csv", "best.pt", "config.json"}:
                yield here / name


def resolve_inputs():
    found = list(input_files()) if not all((DATA_ROOT, CHECKPOINT_PATH, DINOV2_MODEL_DIR)) else []
    root = Path(DATA_ROOT) if DATA_ROOT else find_one(
        (p.parent for p in found if p.name == "test_series.csv"
         if (p.parent / "test.csv").is_file()
         and (p.parent / "sample_submission.csv").is_file()
         and (p.parent / "test_series").is_dir()),
        "competition data directory",
    )
    checkpoint = Path(CHECKPOINT_PATH) if CHECKPOINT_PATH else find_one(
        (p for p in found if p.name == "best.pt"), "best.pt checkpoint"
    )
    model_dir = Path(DINOV2_MODEL_DIR) if DINOV2_MODEL_DIR else find_one(
        (p.parent for p in found if p.name == "config.json"
         if (p.parent / "pytorch_model.bin").is_file()),
        "Hugging Face DINOv2 directory",
    )
    for path in (root / "test.csv", root / "test_series.csv", root / "sample_submission.csv",
                 root / "test_series", checkpoint, model_dir / "config.json",
                 model_dir / "pytorch_model.bin"):
        if not path.exists():
            raise FileNotFoundError(path)
    return root, checkpoint, model_dir


@torch.inference_mode()
def inference_worker(rank, model, local_test_df, series_df, image_size, crop_mm, root, use_metadata, slot_window_budget):
    torch.cuda.set_device(rank)
    device = torch.device(f"cuda:{rank}")
    started = time.monotonic()
    series_df = series_df[series_df.StudyInstanceUID.isin(local_test_df.StudyInstanceUID)]
    print(f"GPU {rank}: indexing {len(series_df)} local series", flush=True)
    series_df = _prepare_series_df(series_df, root / "test_series")
    missing = set(local_test_df.StudyInstanceUID) - set(series_df.StudyInstanceUID)
    if missing:
        raise ValueError(f"No series metadata for {len(missing)} test studies")
    dataset = KneeDataset(local_test_df, series_df, image_size=image_size, crop_mm=crop_mm, slot_window_budget=slot_window_budget)
    print(f"GPU {rank}: {len(dataset)} studies assigned", flush=True)
    loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=NUM_WORKERS, pin_memory=True,
                        persistent_workers=NUM_WORKERS > 0, collate_fn=collate_studies)

    predictions = {}
    for index, batch in enumerate(loader, start=1):
        if index == 1:
            print(f"GPU {rank}: first study loaded after {time.monotonic() - started:.1f}s", flush=True)
        batch = {key: value.to(device, non_blocking=True) if isinstance(value, torch.Tensor) else value
                 for key, value in batch.items()}
        with torch.amp.autocast("cuda", dtype=torch.float16):
            logits = predict_batch(model, batch, use_metadata=use_metadata)
        scores = torch.sigmoid(logits.float()).cpu().numpy()
        for uid, score in zip(batch["study_uid"], scores):
            predictions[str(uid)] = score
        if index % 5 == 0 or index == len(loader):
            print(f"GPU {rank}: {min(index * BATCH_SIZE, len(dataset))}/{len(dataset)} studies, "
                  f"elapsed {time.monotonic() - started:.1f}s", flush=True)

    if len(predictions) != len(dataset):
        raise ValueError(f"Expected {len(dataset)} predictions, got {len(predictions)}")
    values = np.stack([predictions[uid] for uid in local_test_df.StudyInstanceUID])
    if values.shape != (len(dataset), len(LABELS)) or not np.isfinite(values).all():
        raise ValueError("Prediction shape or finiteness check failed")
    if not ((values >= 0) & (values <= 1)).all():
        raise ValueError("Predictions are outside [0, 1]")
    partial = pd.DataFrame(values, columns=LABELS)
    partial.insert(0, "StudyInstanceUID", local_test_df.StudyInstanceUID.to_numpy())
    return partial


def main():
    root, checkpoint_path, model_dir = resolve_inputs()
    print(f"data={root}\ncheckpoint={checkpoint_path}\nDINOv2={model_dir}", flush=True)
    if NUM_GPUS < 1 or NUM_GPUS > torch.cuda.device_count():
        raise RuntimeError(f"Enable a Kaggle GPU accelerator; found {torch.cuda.device_count()} GPU(s).")
    test_df = pd.read_csv(root / "test.csv", dtype={"StudyInstanceUID": str})
    series_df = pd.read_csv(root / "test_series.csv", dtype={
        "StudyInstanceUID": str, "SeriesInstanceUID": str,
    })
    sample = pd.read_csv(root / "sample_submission.csv", dtype={"StudyInstanceUID": str})
    expected = ["StudyInstanceUID", *LABELS]
    if list(sample.columns) != expected:
        raise ValueError(f"Submission columns differ from the training model: {list(sample.columns)}")
    if test_df.StudyInstanceUID.duplicated().any() or sample.StudyInstanceUID.duplicated().any():
        raise ValueError("Duplicate StudyInstanceUID in test or sample_submission")
    if set(test_df.StudyInstanceUID) != set(sample.StudyInstanceUID):
        raise ValueError("test.csv and sample_submission.csv contain different studies")

    print("Loading checkpoint", flush=True)
    checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=False)
    if checkpoint.get("architecture") != ARCHITECTURE:
        raise ValueError("This notebook requires Baseline_v6 weights; checkpoints from other input policies are not accepted")
    state = checkpoint["model"]
    args = checkpoint.get("args")
    if not isinstance(args, dict):
        raise ValueError("Checkpoint must contain training args with image_size, crop_mm, and slot_window_budget")
    missing_args = [name for name in ("image_size", "crop_mm", "slot_window_budget") if name not in args]
    if missing_args:
        raise ValueError(f"Checkpoint is missing preprocessing parameters: {missing_args}")
    slot_window_budget = int(args["slot_window_budget"])
    image_size = int(args["image_size"])
    crop_mm = float(args["crop_mm"])
    if slot_window_budget <= 0 or image_size <= 0 or not np.isfinite(crop_mm) or crop_mm <= 0:
        raise ValueError(f"Invalid preprocessing parameters: slot_window_budget={slot_window_budget}, image_size={image_size}, crop_mm={crop_mm}")
    print(f"Preprocessing: all series, at most {slot_window_budget} raw-adjacent windows per slot, image_size={image_size}, crop_mm={crop_mm:g}", flush=True)
    use_metadata = not bool(args.get("no_metadata", False))
    print(f"Metadata features: {'enabled' if use_metadata else 'disabled'}", flush=True)
    gpu_count = min(NUM_GPUS, len(test_df))
    models = []
    for rank in range(gpu_count):
        model = RSNADINOv2(model_dir=str(model_dir), encoder_chunk_size=ENCODER_CHUNK_SIZE, freeze_backbone=True)
        model.load_state_dict(state, strict=True)
        model.to(torch.device(f"cuda:{rank}")).eval()
        models.append(model)
        print(f"GPU {rank}: model ready", flush=True)
    del checkpoint, state

    # A thread per GPU keeps all computation inside this notebook process.
    # The models and studies are disjoint; no gradient or cross-GPU sync is needed.
    with ThreadPoolExecutor(max_workers=gpu_count) as pool:
        futures = [pool.submit(inference_worker, rank, models[rank],
                               test_df.iloc[rank::gpu_count].reset_index(drop=True),
                               series_df, image_size, crop_mm, root, use_metadata, slot_window_budget)
                   for rank in range(gpu_count)]
        partials = [future.result() for future in futures]
    predictions = pd.concat(partials, ignore_index=True)
    if predictions.StudyInstanceUID.duplicated().any():
        raise ValueError("Duplicate predictions across GPUs")
    if set(predictions.StudyInstanceUID) != set(sample.StudyInstanceUID):
        raise ValueError("GPU predictions do not cover the complete sample submission")
    submission = sample[["StudyInstanceUID"]].merge(predictions, on="StudyInstanceUID",
                                                       how="left", validate="one_to_one")
    if list(submission.columns) != ["StudyInstanceUID", *LABELS]:
        raise ValueError("Incorrect submission column order")
    if not np.isfinite(submission[LABELS].to_numpy()).all():
        raise ValueError("Non-finite value in merged submission")
    OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
    submission.to_csv(OUTPUT_PATH, index=False)
    print(f"Saved {OUTPUT_PATH}: {submission.shape[0]} rows, {submission.shape[1]} columns")
    print(submission.head(3).to_string(index=False))


if __name__ == "__main__":
    main()
